In [1]:
import re
import json
from typing import Dict, List, Optional
from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')
path1='/content/drive/MyDrive/Food additives RAG+LLM/2.10.2 and 2.10.3/2.10.2 and 2.10.3.txt'
with open(path1, 'r') as file:
        content = file.read()
def parse_food_analysis_output(content: str) -> Dict:
    """
    Parse the food additive analysis console output and extract structured information.

    Args:
        content (str): The raw console output string

    Returns:
        Dict: Structured data containing all extracted information
    """

    result = {
        'entry_info': {},
        'ingredient_list': '',
        'top_3_predictions': [],
        'selected_food_category': '',
        'identified_additives': [],
        'complete_llm_response': ''
    }

    # 1. Extract Entry Information
    entry_match = re.search(r'📝 Random sample loaded:\s*\n\s*Entry ID: (\d+)\s*\n\s*Product: ([^\n]+)\s*\n\s*Brand: ([^\n]+)\s*\n\s*Category: ([^\n]+)\s*\n.*?Brand Name: ([^\n]+)', content, re.DOTALL)

    if entry_match:
        result['entry_info'] = {
            'entry_id': entry_match.group(1),
            'product': entry_match.group(2),
            'brand': entry_match.group(3),
            'category': entry_match.group(4),
            'brand_name': entry_match.group(5)
        }

    # 2. Extract Original Ingredient List
    # Look for the original ingredient text in the MLX debug section
    ingredient_match = re.search(r'📝 MLX Identify: ORIGINAL INGREDIENT TEXT:\s*\n={40}\s*\n(.*?)\n={40}', content, re.DOTALL)
    if ingredient_match:
        result['ingredient_list'] = ingredient_match.group(1).strip()

    # 3. Extract Top 3 Predictions
    predictions_section = re.search(r'Top 3 predictions:\s*\n(.*?)\n🔍', content, re.DOTALL)
    if predictions_section:
        predictions_text = predictions_section.group(1)
        prediction_matches = re.findall(r'(\d+)\.\s*([^:]+):\s*([\d.]+)', predictions_text)

        for match in prediction_matches:
            result['top_3_predictions'].append({
                'rank': int(match[0]),
                'category': match[1].strip(),
                'confidence': float(match[2])
            })

    # 4. Extract Selected Food Category
    category_match = re.search(r'📝 Selected food category: ([^\n]+)', content)
    if category_match:
        result['selected_food_category'] = category_match.group(1).strip()

    # 5. Extract Identified Additives
    additives_dict = {}  # Use dict to track case-insensitive duplicates while preserving original case

    def add_additive(original_text):
        """Add additive while checking for case-insensitive duplicates."""
        key = original_text.upper().strip()  # Use uppercase for comparison
        if key not in additives_dict:
            additives_dict[key] = original_text.strip()  # Store original case

    # Find all direct matches
    direct_matches = re.findall(r'✅ Direct match found: \'([^\']+)\' -> \'([^\']+)\'', content)
    for match in direct_matches:
        add_additive(match[0])

    # Find all direct lookups
    direct_lookups = re.findall(r'✅ Found via direct lookup: \'([^\']+)\'', content)
    for match in direct_lookups:
        add_additive(match)

    # Find all embedding searches
    embedding_searches = re.findall(r'✅ Found via embedding search: \'([^\']+)\'', content)
    for match in embedding_searches:
        add_additive(match)

    # Also extract from Native search results
    native_matches = re.findall(r'✅ Native: Found knowledge for \'([^\']+)\' -> \'([^\']+)\'', content)
    for match in native_matches:
        add_additive(match[0])

    # Extract additives from MLX LLM identification response
    llm_additives_match = re.search(r'🤖 MLX Additive Identification: COMPLETE LLM RESPONSE:\s*\n={40}\s*\n(.*?)\n={40}', content, re.DOTALL)
    if llm_additives_match:
        llm_additives_text = llm_additives_match.group(1).strip()
        llm_additives = [line.strip() for line in llm_additives_text.split('\n') if line.strip()]
        for additive in llm_additives:
            add_additive(additive)

    result['identified_additives'] = sorted(list(additives_dict.values()))

    # 6. Extract Complete LLM Response
    llm_response_match = re.search(r'🤖 MLX Agent 3 \(Explanation\): COMPLETE LLM RESPONSE:\s*\n={40}\s*\n(.*?)\n={40}', content, re.DOTALL)
    if llm_response_match:
        result['complete_llm_response'] = llm_response_match.group(1).strip()

    return result

def print_parsed_results(parsed_data: Dict) -> None:
    """
    Print the parsed results in a readable format.
    """

    print("=" * 60)
    print("PARSED FOOD ANALYSIS RESULTS")
    print("=" * 60)

    # Entry Information
    print("\n📋 ENTRY INFORMATION:")
    entry_info = parsed_data['entry_info']
    if entry_info:
        print(f"  Entry ID: {entry_info.get('entry_id', 'N/A')}")
        print(f"  Product: {entry_info.get('product', 'N/A')}")
        print(f"  Brand: {entry_info.get('brand', 'N/A')}")
        print(f"  Category: {entry_info.get('category', 'N/A')}")
        print(f"  Brand Name: {entry_info.get('brand_name', 'N/A')}")
    else:
        print("  No entry information found")

    # Ingredient List
    print(f"\n🧾 ORIGINAL INGREDIENT LIST:")
    ingredient_list = parsed_data['ingredient_list']
    if ingredient_list:
        print(f"  {ingredient_list}")
    else:
        print("  No ingredient list found")

    # Top 3 Predictions
    print(f"\n🎯 TOP 3 PREDICTIONS:")
    predictions = parsed_data['top_3_predictions']
    if predictions:
        for pred in predictions:
            print(f"  {pred['rank']}. {pred['category']}: {pred['confidence']:.3f}")
    else:
        print("  No predictions found")

    # Selected Category
    print(f"\n✅ SELECTED FOOD CATEGORY:")
    selected_category = parsed_data['selected_food_category']
    if selected_category:
        print(f"  {selected_category}")
    else:
        print("  No selected category found")

    # Identified Additives
    print(f"\n🔍 IDENTIFIED ADDITIVES ({len(parsed_data['identified_additives'])}):")
    additives = parsed_data['identified_additives']
    if additives:
        for i, additive in enumerate(additives, 1):
            print(f"  {i}. {additive}")
    else:
        print("  No additives identified")

    # Complete LLM Response
    print(f"\n🤖 COMPLETE LLM RESPONSE:")
    llm_response = parsed_data['complete_llm_response']
    if llm_response:
        print(f"  {llm_response}")
    else:
        print("  No LLM response found")

def save_to_json(parsed_data: Dict, filename: str = 'parsed_food_analysis.json') -> None:
    """
    Save parsed data to a JSON file.
    """
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(parsed_data, f, indent=2, ensure_ascii=False)
    print(f"\n💾 Data saved to {filename}")

Mounted at /content/drive


In [2]:
contents = content.split('📝 Random sample lo')[1:]
print(len(contents))
for i in  range(len(contents)):
  contents[i] = '📝 Random sample lo' + contents[i]

108


In [7]:
a=set()
for i in  range(len(contents)):
  parsed_results = parse_food_analysis_output(contents[i])
  a.add(parsed_results['entry_info']['entry_id'])
  print("=" * 60)
  print(f"#{i+1}")
  print_parsed_results(parsed_results)

#1
PARSED FOOD ANALYSIS RESULTS

📋 ENTRY INFORMATION:
  Entry ID: 4792
  Product: FRENCH TOAST STICKS CINNAMON
  Brand: Weis Markets, Inc.
  Category: Frozen Pancakes, Waffles, French Toast & Crepes
  Brand Name: WEIS QUALITY

🧾 ORIGINAL INGREDIENT LIST:
  ENRICHED BREAD (ENRICHED WHEAT FLOUR [WHEAT FLOUR, MALTED BARLEY FLOUR, REDUCED IRON, NIACIN, THIAMINE MONONITRATE, RIBOFLAVIN, FOLIC ACID], WATER, HIGH FRUCTOSE CORN SYRUP, YEAST, SOYBEAN OIL, SALT, CALCIUM PROPIONATE [PRESERVATIVE], DATEM, MONOGLYCERIDES, SPICE AND COLORING, SOY LECITHIN), WATER, BATTER (BLEACHED WHEAT FLOUR, SUGAR, DEXTROSE, DRIED EGGS, YELLOW CORN FLOUR, CORN SYRUP SOLIDS, NATURAL FLAVOR, MODIFIED CORNSTARCH, SALT, LEAVENING [SODIUM ALUMINUM PHOSPHATE, SODIUM BICARBONATE], NONFAT DRY MILK, SPICE, ARTIFICIAL FLAVOR, CARBOHYDRATE GUM, SILICON DIOXIDE [ADDED TO PREVENT CAKING]), SOYBEAN OIL, CINNAMON SUGAR (SUGAR, CINNAMON, SILICON DIOXIDE [ADDED TO PREVENT CAKING]).

🎯 TOP 3 PREDICTIONS:
  1. Frozen Bread & Dough: 

In [8]:
print(a)

{'30980', '28502', '22477', '8975', '22427', '26218', '523', '31025', '28602', '28287', '29511', '21954', '13256', '6672', '9937', '12551', '6424', '23918', '40946', '29296', '33057', '22552', '1904', '5332', '32984', '30313', '30460', '26019', '214', '36368', '25603', '13102', '26569', '34917', '16221', '29980', '6256', '1867', '41398', '27476', '31548', '41685', '25538', '19377', '4250', '10127', '16535', '32462', '8250', '19398', '19213', '3743', '3598', '19650', '3679', '4792', '10019', '31189', '40732', '28109', '30014', '19518', '18838', '37385', '22909', '24792', '36851', '2261', '41449', '7216', '38339', '15601', '16856', '9125', '5487', '299', '16827', '30478', '11999', '31414', '7767', '15004', '15100', '4721', '32938', '1327', '33109', '15537', '18987', '31571', '40999', '26660', '32883', '5965', '9814', '26107', '4457', '30776', '41998', '1854', '29729', '11566', '14760', '6745', '11608', '33980', '1344', '25496'}


In [9]:
print(len(a))

108
